# Sage on simulated DDA — true FDR / TPR

End-to-end demo of running Sage on a TimSim-simulated `.d` and scoring
the result against the ground-truth `synthetic_data.db`.

Adjust the paths in the next cell to your local layout.

In [ ]:
from pathlib import Path

SAGE_PARQUET = Path('runs/ci-smoke/results.sage.parquet')
TRUTH_DBS = [
    Path('data/ci-smoke/SAGEBENCH-CI-HELA-SMOKE-001.d/synthetic_data.db'),
    Path('data/ci-smoke/SAGEBENCH-CI-HELA-SMOKE-002.d/synthetic_data.db'),
]

In [ ]:
from sagebench import ground_truth, sage_output, metrics

truth = ground_truth.union(*[ground_truth.load(p) for p in TRUTH_DBS])
hits  = sage_output.load(SAGE_PARQUET)

print(f'truth: {truth.n_targets:,} (sequence, charge) pairs')
print(f'sage:  {hits.n_psms:,} PSMs ({hits.n_targets:,} target / {hits.n_psms - hits.n_targets:,} decoy)')

In [ ]:
metrics.sweep(hits, truth, q_cutoffs=[0.001, 0.005, 0.01, 0.05])

In [ ]:
import matplotlib.pyplot as plt

roc = metrics.roc_points(hits, truth, n_points=200)
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(roc['true_fdr'], roc['tpr'])
ax.set_xlabel('true FDR')
ax.set_ylabel('TPR')
ax.set_xlim(0, 0.1)
ax.grid(alpha=0.3)
ax.set_title('Sage on SAGEBench HeLa CI smoke')